In [1]:
from pathlib import Path
from references import functions, constants

workdir = Path('/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even')
resultsdir = workdir / 'results'

files = functions.get_mc_files(resultsdir)

for file in files:
    print(file)

/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/TTbar_dl_2022.root
/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/TTbar_sl_2022.root
/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/bbWW_dl_2022.root
/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/bbWW_sl_2022.root
/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/tWminus_dl_2022.root
/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/tWminus_sl_2022.root
/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/tbarWplus_dl_2022.root
/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even/results/tbarWplus_sl_2022.root


In [2]:
import ROOT
import re

def change_histogram_names(root_file_path):
    """
    Change histogram names in a ROOT file according to specific patterns.
    
    Pattern 1: SL_4j_resolved_<model>_<class> -> SL_4j_resolved___<model>__<class>
    Pattern 2: SL_4j_resolved_<model>_<class>_sub -> SL_4j_resolved__<class>___<model>__<class>
    """
    
    # Open the ROOT file in UPDATE mode to modify it
    root_file = ROOT.TFile.Open(root_file_path, "UPDATE")
    
    if not root_file or root_file.IsZombie():
        print(f"Error: Could not open file {root_file_path}")
        return
    
    # Get list of all keys (histogram names) in the file
    keys = root_file.GetListOfKeys()
    
    # Store original names and new names
    rename_map = {}
    
    # Process each histogram
    for key in keys:
        old_name = key.GetName()
        new_name = transform_histogram_name(old_name)
        
        if new_name != old_name:
            rename_map[old_name] = new_name
            print(f"Will rename: {old_name} -> {new_name}")
    
    # Perform the renaming
    for old_name, new_name in rename_map.items():
        # Get the histogram object
        hist = root_file.Get(old_name)
        if hist:
            # Set the new name
            hist.SetName(new_name)
            hist.SetTitle(new_name)  # Also update title if desired
            
            # Write the histogram with the new name
            hist.Write(new_name, ROOT.TObject.kOverwrite)
            
            # Delete the old histogram from the file
            root_file.Delete(f"{old_name};*")
            
            print(f"Renamed: {old_name} -> {new_name}")
        else:
            print(f"Warning: Could not find histogram {old_name}")
    
    # Save changes and close file
    root_file.Write()
    root_file.Close()
    
    print(f"Successfully updated {len(rename_map)} histogram names in {root_file_path}")

def transform_histogram_name(name):
    """
    Transform histogram names according to the specified patterns.
    
    Pattern 1: SL_4j_resolved_<model>_<class> -> SL_4j_resolved___<model>__<class>
    Pattern 2: SL_4j_resolved_<model>_<class>_sub -> SL_4j_resolved__<class>___<model>__<class>
    """
    
    # Pattern for: SL_4j_resolved_<model>_<class>_sub
    pattern1 = r'^(SL_4j_resolved)_(.+?)_([^_]+)_sub$'
    match1 = re.match(pattern1, name)
    
    if match1:
        prefix, model, nn_class = match1.groups()
        # Transform to: SL_4j_resolved__<class>___<model>__<class>
        new_name = f"{prefix}__{nn_class}___{model}__{nn_class}"
        return new_name
    
    # Pattern for: SL_4j_resolved_<model>_<class>
    pattern2 = r'^(SL_4j_resolved)_(.+?)_([^_]+)$'
    match2 = re.match(pattern2, name)
    
    if match2:
        prefix, model, nn_class = match2.groups()
        # Transform to: SL_4j_resolved___<model>__<class>
        new_name = f"{prefix}___{model}__{nn_class}"
        return new_name
    
    # Return original name if no pattern matches
    return name

def preview_changes(root_file_path):
    """
    Preview what changes would be made without actually modifying the file.
    """
    root_file = ROOT.TFile.Open(root_file_path, "READ")
    
    if not root_file or root_file.IsZombie():
        print(f"Error: Could not open file {root_file_path}")
        return
    
    keys = root_file.GetListOfKeys()
    changes_found = False
    
    print("Preview of changes:")
    print("-" * 60)
    
    for key in keys:
        old_name = key.GetName()
        new_name = transform_histogram_name(old_name)
        
        if new_name != old_name:
            print(f"{old_name} -> {new_name}")
            changes_found = True
    
    if not changes_found:
        print("No histograms match the renaming patterns.")
    
    root_file.Close()

Welcome to JupyROOT 6.30/02


In [4]:
preview_changes(str(files[0]))

Preview of changes:
------------------------------------------------------------
SL_4j_resolved_multi_HH_ttbar_tW_both_HH -> SL_4j_resolved___multi_HH_ttbar_tW_both__HH
SL_4j_resolved_multi_HH_ttbar_tW_both_HH_sub -> SL_4j_resolved__HH___multi_HH_ttbar_tW_both__HH
SL_4j_resolved_multi_HH_ttbar_tW_both_TTbar -> SL_4j_resolved___multi_HH_ttbar_tW_both__TTbar
SL_4j_resolved_multi_HH_ttbar_tW_both_TTbar_sub -> SL_4j_resolved__TTbar___multi_HH_ttbar_tW_both__TTbar
SL_4j_resolved_multi_HH_ttbar_tW_both_tW -> SL_4j_resolved___multi_HH_ttbar_tW_both__tW
SL_4j_resolved_multi_HH_ttbar_tW_both_tW_sub -> SL_4j_resolved__tW___multi_HH_ttbar_tW_both__tW
SL_4j_resolved_multi_HH_ttbar_tW_lrs_HH -> SL_4j_resolved___multi_HH_ttbar_tW_lrs__HH
SL_4j_resolved_multi_HH_ttbar_tW_lrs_HH_sub -> SL_4j_resolved__HH___multi_HH_ttbar_tW_lrs__HH
SL_4j_resolved_multi_HH_ttbar_tW_lrs_TTbar -> SL_4j_resolved___multi_HH_ttbar_tW_lrs__TTbar
SL_4j_resolved_multi_HH_ttbar_tW_lrs_TTbar_sub -> SL_4j_resolved__TTbar___multi_

In [5]:
for file in files:
    change_histogram_names(str(file))

Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_HH -> SL_4j_resolved___multi_HH_ttbar_tW_both__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_HH_sub -> SL_4j_resolved__HH___multi_HH_ttbar_tW_both__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_TTbar -> SL_4j_resolved___multi_HH_ttbar_tW_both__TTbar
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_TTbar_sub -> SL_4j_resolved__TTbar___multi_HH_ttbar_tW_both__TTbar
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_tW -> SL_4j_resolved___multi_HH_ttbar_tW_both__tW
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_tW_sub -> SL_4j_resolved__tW___multi_HH_ttbar_tW_both__tW
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_lrs_HH -> SL_4j_resolved___multi_HH_ttbar_tW_lrs__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_lrs_HH_sub -> SL_4j_resolved__HH___multi_HH_ttbar_tW_lrs__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_lrs_TTbar -> SL_4j_resolved___multi_HH_ttbar_tW_lrs__TTbar
Will rename: SL_4j_resolved_multi_HH_ttbar_t